# Logit lens on GPT-2

Project the residual stream after every layer through the final LayerNorm and unembedding,
and look at the top-1 token at each position — to see at what depth the model settles on its prediction.

## Setup

In [1]:
!git clone https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git

Cloning into 'jacobian-lens-gpt2'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 71 (delta 8), reused 15 (delta 4), pack-reused 38 (from 1)
Receiving objects: 100% (71/71), 1.83 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [2]:
import json
import sys

import torch
from IPython.display import display

REPO_DIR = '/kaggle/working/jacobian-lens-gpt2'
sys.path.append(REPO_DIR)

from gpt2 import GPT2LensModel
from jlens import JacobianLens
from jlens.vis import build_page, compute_slice, notebook_iframe

In [3]:
gpt = GPT2LensModel.from_pretrained()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Dataset

In [4]:
dataset_name = REPO_DIR + "/data/evaluations/lens-eval-order-ops.json"

dataset = json.load(open(dataset_name))

In [5]:
dataset['items'][0]

{'name': 'parens-add-mult',
 'prompt': '(2 + 3) * 4 = ',
 'target': '20',
 'intermediates': ['5', 'multiplication']}

## Sanity checks

Shapes of the encoded prompt and the hidden states, and the lens prediction at a single position.

In [6]:
ids = gpt.encode(dataset['items'][0]['prompt'])
with torch.no_grad():
    hidden_states = gpt.forward(ids, output_hidden_states=True).hidden_states

In [7]:
ids.shape

torch.Size([1, 10])

In [8]:
hidden_states[0].shape

torch.Size([1, 10, 768])

In [9]:
gpt.unembed(hidden_states[0][0, 1, :]).argmax()

tensor(7)

In [10]:
vocab = gpt.tokenizer.get_vocab()
token_to_item = {
    vocab[i] : i
    for i in vocab
}

## Logit lens visualisation

Reuses the slice page from `walkthrough.ipynb` (`compute_slice` + `build_page`): an interactive
position × layer view of token ranks. The logit lens is a Jacobian lens with `J_l = I` on every layer,
so each block output goes straight through `ln_f` + unembedding.

In [11]:
# логит-линза = J-линза с единичной матрицей на каждом слое
logit_lens = JacobianLens(
    {layer: torch.eye(gpt.d_model) for layer in range(gpt.n_layers - 1)},
    n_prompts=0,
    d_model=gpt.d_model,
)


def show_logit_lens(prompt, title="Logit lens", description=""):
    slice_data = compute_slice(gpt, logit_lens, prompt)
    page, _, _ = build_page(slice_data, prompt, title=title, description=description)
    display(notebook_iframe(page))

### Dataset prompts

In [12]:
for sample in dataset['items'][0:5]:
    show_logit_lens(
        sample['prompt'].rstrip(),
        title=sample['name'],
        description=f"target: {sample['target']!r}, intermediates: {sample['intermediates']}",
    )

### Custom prompt

In [13]:
show_logit_lens("The Eifflel Tower is located in the city of")